# C. YOLO 6종 결과 합치기

`result_*.json`을 읽어 split 일치 여부를 먼저 검사한 뒤, 6행 비교표와 그래프 3장을 만듭니다. 미제출 모델도 표에 남습니다.


In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = Path('/content/drive/MyDrive/shaft_sweep')
RESULTS = DRIVE / 'results'
MODELS = ['yolo26n','yolo26s','yolo26m','yolo11n','yolo11s','yolo11m']
RESULTS.mkdir(parents=True, exist_ok=True)
files = sorted(RESULTS.glob('result_*.json'))
print('찾은 결과:', len(files), '개')
if not files: raise FileNotFoundError('result_*.json이 없습니다. 팀원 결과를 results/에 모아 주세요.')


In [ ]:
# 1) JSON 로드 + split 교차 검증
loaded = {}
for path in files:
    data = json.loads(path.read_text(encoding='utf-8'))
    key = Path(data.get('model', path.stem.removeprefix('result_'))).stem
    if key not in MODELS:
        warnings.warn(f'비교 대상이 아닌 결과는 제외합니다: {path.name} ({key})')
        continue
    if key in loaded:
        raise RuntimeError(f'{key} 결과가 중복입니다. 파일을 하나만 남기세요.')
    loaded[key] = data

split_signatures = {}
for model, data in loaded.items():
    split = data.get('split', {})
    signature = (tuple(split.get('train_specimens', [])), tuple(split.get('val_specimens', [])))
    split_signatures[model] = signature
unique_splits = set(split_signatures.values())
SPLIT_OK = len(unique_splits) == 1 and all(sig[0] and sig[1] for sig in unique_splits)
if not SPLIT_OK:
    warnings.warn('⚠️ split 불일치: 누군가 다른 데이터셋/split을 썼습니다. 해당 결과는 공정 비교 대상이 아닙니다.', RuntimeWarning)
    display(pd.DataFrame([{'model':m,'train_specimens':list(s[0]),'val_specimens':list(s[1])} for m,s in split_signatures.items()]))
else:
    print('✅ 제출된 모든 결과의 train/val 시편 split이 같습니다.')


In [ ]:
# 2) 항상 6행인 비교표 (미제출 포함)
def pick(d, *keys):
    for key in keys: d = d.get(key, {}) if isinstance(d, dict) else {}
    return d if d != {} else np.nan

rows = []
for model in MODELS:
    d = loaded.get(model)
    if d is None:
        rows.append({'model':model, 'status':'미제출'})
        continue
    rows.append({
        'model':model, 'status':'제출',
        'mAP50':pick(d,'detect','map50'), 'mAP50-95':pick(d,'detect','map50_95'),
        'precision':pick(d,'detect','precision'), 'recall':pick(d,'detect','recall'),
        'infer_ms':pick(d,'detect','infer_ms'), 'params':pick(d,'train','params'),
        'GFLOPs':pick(d,'train','gflops'), 'batch_used':pick(d,'train','batch_used'),
        'calibration_pixel':pick(d,'calib','calibration_pixel'), 'mm_per_pixel':pick(d,'calib','mm_per_pixel'),
        'bias_um':pick(d,'measure','bias_um'), 'MAE_um':pick(d,'measure','mae_um'),
        'RMSE_um':pick(d,'measure','rmse_um'), 'repeat_sigma_um':pick(d,'measure','repeat_sigma_um'),
        'box_h_std_px':pick(d,'measure','box_h_std_px'), 'box_y1_std_px':pick(d,'measure','box_y1_std_px'),
        'box_y2_std_px':pick(d,'measure','box_y2_std_px'), 'ng_detected':pick(d,'measure','ng_detected'),
        'false_alarm':pick(d,'measure','false_alarm'), 'n_ok':pick(d,'robust','n_ok'),
        'n_no_roi':pick(d,'robust','n_no_roi'), 'n_insufficient_edges':pick(d,'robust','n_insufficient_edges'),
        'n_outlier_reject':pick(d,'robust','n_outlier_reject'),
    })
summary = pd.DataFrame(rows)
summary.to_csv(RESULTS/'summary.csv', index=False, encoding='utf-8-sig')
display(summary)


In [ ]:
# 3) 그래프 3장 — 미제출/계산 불가는 회색 또는 빈 값으로 표시
plt.style.use('seaborn-v0_8-whitegrid')
colors = ['#4C78A8' if s=='제출' else '#D3D3D3' for s in summary.status]
def bar_chart(column, ylabel, title, filename):
    values = pd.to_numeric(summary[column], errors='coerce')
    fig, ax = plt.subplots(figsize=(9,5))
    ax.bar(summary.model, values.fillna(0), color=colors)
    for i,v in enumerate(values):
        ax.text(i, 0 if pd.isna(v) else v, '미제출' if summary.status.iloc[i]=='미제출' else ('' if pd.isna(v) else f'{v:.3f}'), ha='center', va='bottom', fontsize=9)
    ax.set(ylabel=ylabel, title=title); fig.tight_layout(); fig.savefig(RESULTS/filename, dpi=180); plt.show()

bar_chart('mAP50-95', 'mAP50-95', '모델별 탐지 성능', '01_map50_95.png')
bar_chart('repeat_sigma_um', '시편별 반복 σ 평균 (µm, 낮을수록 좋음)', '모델별 측정 재현성 — 최종 결론 지표', '02_repeat_sigma_um.png')

bias = pd.to_numeric(summary.bias_um, errors='coerce')
mae = pd.to_numeric(summary.MAE_um, errors='coerce')
fig, ax = plt.subplots(figsize=(9,5))
valid = bias.notna() & mae.notna()
ax.errorbar(np.arange(len(summary))[valid], bias[valid], yerr=mae[valid], fmt='o', capsize=6, color='#F58518')
ax.axhline(0, color='black', linewidth=1); ax.set_xticks(range(len(summary)), summary.model)
ax.set(ylabel='bias ± MAE (µm)', title='모델별 측정 오차'); fig.tight_layout(); fig.savefig(RESULTS/'03_bias_mae.png', dpi=180); plt.show()


In [ ]:
# 4) 사람이 바로 공유할 수 있는 Markdown 보고서
submitted = summary[summary.status.eq('제출')].copy()
repeatable = submitted.dropna(subset=['repeat_sigma_um'])
winner = repeatable.loc[repeatable.repeat_sigma_um.astype(float).idxmin(), 'model'] if len(repeatable) else '판정 불가'
missing = summary.loc[summary.status.eq('미제출'), 'model'].tolist()
split_text = '일치' if SPLIT_OK else '불일치 — 공정 비교 불가, 원본 JSON 확인 필요'
table_cols = ['model','status','mAP50-95','bias_um','MAE_um','RMSE_um','repeat_sigma_um','ng_detected','false_alarm']
try:
    table_md = summary[table_cols].to_markdown(index=False)
except ImportError:
    table_md = summary[table_cols].to_csv(index=False)
report = f'''# 샤프트 외경 측정 YOLO 모델 비교

## 검증 상태

- 제출: {len(submitted)}/6개 모델
- 미제출: {', '.join(missing) if missing else '없음'}
- split 교차 검증: **{split_text}**
- 표본: 전체 시편 13개 / 이미지 161장 (오차 평가는 val 시편만)

## 비교표

{table_md}

## 잠정 결론

측정 재현성 σ가 가장 낮은 모델은 **{winner}**입니다. mAP 1등과 다르면 프로젝트 목적상 재현성 1등을 우선합니다. 6개가 모두 제출되고 split이 일치할 때만 최종 결론으로 확정하세요.

## Limitations

- 라벨은 기존 best.pt의 pre-annotation을 사람이 검수한 것이므로, 이 비교는 “기존 박스 규칙을 어느 아키텍처가 가장 정밀하게 재현하는가”에 답합니다. 박스 규칙 자체의 타당성을 독립적으로 검증하지 않습니다.
- 시편 13개/이미지 161장의 작은 표본이므로 수치와 함께 표본 수를 제시해야 합니다.
- 캘리브레이션 이미지는 각 모델의 mm/pixel 산출에만 사용했고 val 오차 평가에서는 제외했습니다.

## 그래프

1. `01_map50_95.png` — 탐지 지표
2. `02_repeat_sigma_um.png` — 측정 재현성(최종 결론 지표)
3. `03_bias_mae.png` — bias ± MAE
'''
(RESULTS/'report.md').write_text(report, encoding='utf-8')
print('완료:', RESULTS/'summary.csv', RESULTS/'report.md')
